In [1]:
import pandas as pd
import numpy as np
from pymining import seqmining
import matplotlib.pyplot as plt

In [2]:
# Step 1: Load the datasets
orders = pd.read_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\instacart\orders.csv')
products = pd.read_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\instacart\products.csv')
order_products__prior = pd.read_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\instacart\order_products__prior.csv')
order_products__train = pd.read_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\instacart\order_products__train.csv')
aisles = pd.read_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\instacart\aisles.csv')
departments = pd.read_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\instacart\departments.csv')

print("All datasets are loaded successfully")

All datasets are loaded successfully


### Null-Value Treatment

In [3]:
# Check for missing values in each dataset
print("Missing values in orders:")
print(orders.isnull().sum())
print("Missing values in products:")
print(products.isnull().sum())
print("Missing values in order_products__prior:")
print(order_products__prior.isnull().sum())
print("Missing values in order_products__train:")
print(order_products__train.isnull().sum())
print("Missing values in aisles:")
print(aisles.isnull().sum())
print("Missing values in departments:")
print(departments.isnull().sum())

Missing values in orders:
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64
Missing values in products:
product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64
Missing values in order_products__prior:
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64
Missing values in order_products__train:
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64
Missing values in aisles:
aisle_id    0
aisle       0
dtype: int64
Missing values in departments:
department_id    0
department       0
dtype: int64


In [4]:
# For orders, fill missing days_since_prior_order with 0 (assuming first orders)
orders['days_since_prior_order'] = orders['days_since_prior_order'].fillna(0)
print("Missing values in orders after handling:")
print(orders.isnull().sum())

Missing values in orders after handling:
order_id                  0
user_id                   0
eval_set                  0
order_number              0
order_dow                 0
order_hour_of_day         0
days_since_prior_order    0
dtype: int64


### Convert Data Types

In [5]:
# Convert order-related numeric fields to integer
orders = orders.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16', 
                        'order_dow': 'int8', 'order_hour_of_day': 'int8', 'days_since_prior_order': 'float32'})

# Convert categorical fields
orders['eval_set'] = orders['eval_set'].astype('category')

# Convert product-related fields
products = products.astype({'product_id': 'int32', 'aisle_id': 'int16', 'department_id': 'int16'})
aisles = aisles.astype({'aisle_id': 'int16'})
departments = departments.astype({'department_id': 'int16'})

# Convert order-product mapping fields
order_products__prior = order_products__prior.astype({'order_id': 'int32', 'product_id': 'int32', 
                                                      'add_to_cart_order': 'int16', 'reordered': 'int8'})
order_products__train = order_products__train.astype({'order_id': 'int32', 'product_id': 'int32', 
                                                      'add_to_cart_order': 'int16', 'reordered': 'int8'})

print("Data type conversion complete!")

Data type conversion complete!


### Data Integrity Check

In [6]:
# Ensure every product in order_products exists in products
valid_product_ids = set(products['product_id'])
order_products__prior = order_products__prior[order_products__prior['product_id'].isin(valid_product_ids)]
order_products__train = order_products__train[order_products__train['product_id'].isin(valid_product_ids)]
print("Data integrity check complete!")

Data integrity check complete!


### Merge Orders with Order Products to Bring User Information

In [7]:
# Merge prior orders with orders to get user_id and other order details
orders_prior = order_products__prior.merge(orders, on="order_id", how="left")
# (We could also merge the train orders similarly if needed)
print("Merged orders with products (sample):")
print(orders_prior.head())

Merged orders with products (sample):
   order_id  product_id  add_to_cart_order  reordered  user_id eval_set  \
0         2       33120                  1          1   202279    prior   
1         2       28985                  2          1   202279    prior   
2         2        9327                  3          0   202279    prior   
3         2       45918                  4          1   202279    prior   
4         2       30035                  5          0   202279    prior   

   order_number  order_dow  order_hour_of_day  days_since_prior_order  
0             3          5                  9                     8.0  
1             3          5                  9                     8.0  
2             3          5                  9                     8.0  
3             3          5                  9                     8.0  
4             3          5                  9                     8.0  


In [8]:
# For sequential mining, we need the user_id, so we merge orders (which has user_id) with order_products__prior.
order_products_with_user = order_products__prior.merge(orders[['order_id', 'user_id']], on='order_id', how='left')
print("Order-products with user info (sample):")
print(order_products_with_user.head())

Order-products with user info (sample):
   order_id  product_id  add_to_cart_order  reordered  user_id
0         2       33120                  1          1   202279
1         2       28985                  2          1   202279
2         2        9327                  3          0   202279
3         2       45918                  4          1   202279
4         2       30035                  5          0   202279


###  Create User Purchase Sequences

In [9]:
# Group by user_id to create a sequence (list) of product_ids for each user.
user_sequences = order_products_with_user.groupby('user_id')['product_id'].apply(list).reset_index()
print("User sequences (first few rows):")
print(user_sequences.head())

User sequences (first few rows):
   user_id                                         product_id
0        1  [196, 12427, 10258, 25133, 10326, 17122, 41787...
1        2  [49451, 32792, 32139, 34688, 36735, 37646, 228...
2        3  [38596, 21903, 248, 40604, 8021, 17668, 21137,...
3        4  [22199, 25146, 1200, 17769, 43704, 37646, 1186...
4        5  [27344, 24535, 43693, 40706, 16168, 21413, 139...


In [10]:
# For SPM, create a space-separated string of product IDs per user.
user_sequences['sequence'] = user_sequences.groupby('user_id')['product_id'].transform(lambda x: ' '.join(map(str, x)))
print("User sequences with sequence string (sample):")
print(user_sequences.head())

User sequences with sequence string (sample):
   user_id                                         product_id  \
0        1  [196, 12427, 10258, 25133, 10326, 17122, 41787...   
1        2  [49451, 32792, 32139, 34688, 36735, 37646, 228...   
2        3  [38596, 21903, 248, 40604, 8021, 17668, 21137,...   
3        4  [22199, 25146, 1200, 17769, 43704, 37646, 1186...   
4        5  [27344, 24535, 43693, 40706, 16168, 21413, 139...   

                                            sequence  
0  [196, 12427, 10258, 25133, 10326, 17122, 41787...  
1  [49451, 32792, 32139, 34688, 36735, 37646, 228...  
2  [38596, 21903, 248, 40604, 8021, 17668, 21137,...  
3  [22199, 25146, 1200, 17769, 43704, 37646, 1186...  
4  [27344, 24535, 43693, 40706, 16168, 21413, 139...  


### Convert Sequences to a Format Suitable for Mining

In [11]:
# Convert each sequence string into a list of product IDs (as strings)
sequences_for_mining = [seq.split() for seq in user_sequences['sequence']]
print("First 5 sequences for mining:")
print(sequences_for_mining[:5])

First 5 sequences for mining:
[['[196,', '12427,', '10258,', '25133,', '10326,', '17122,', '41787,', '13176,', '196,', '12427,', '10258,', '25133,', '30450,', '196,', '10258,', '12427,', '25133,', '13032,', '196,', '12427,', '10258,', '25133,', '26405,', '49235,', '46149,', '25133,', '196,', '10258,', '12427,', '196,', '10258,', '12427,', '13176,', '26088,', '13032,', '196,', '14084,', '12427,', '26088,', '26405,', '196,', '46149,', '39657,', '38928,', '25133,', '10258,', '35951,', '13032,', '12427,', '12427,', '196,', '10258,', '25133,', '46149,', '49235,', '196,', '12427,', '10258,', '25133]'], ['[49451,', '32792,', '32139,', '34688,', '36735,', '37646,', '22829,', '24852,', '47209,', '33276,', '45613,', '9681,', '21150,', '32792,', '28918,', '24852,', '34688,', '48110,', '47209,', '32052,', '17758,', '40198,', '46886,', '22963,', '23,', '20084,', '2002,', '5212,', '14306,', '24852,', '16589,', '1559,', '19156,', '18523,', '22825,', '27413,', '33754,', '21709,', '47209,', '7781,', '2

### Define Utility Based on Available Variables

### Apply Sequential Pattern Mining Using pymining

In [ ]:
# Sample a portion of the data, e.g., 10% of users
sampled_user_sequences = user_sequences.sample(frac=0.1, random_state=42)

# Create sequences for mining from the sampled data
sequences_for_mining = sampled_user_sequences['product_id'].tolist()

# Continue with creating sparse matrix as before
unique_products = products['product_id'].unique()
num_users = len(sampled_user_sequences)
num_products = len(unique_products)

# Create sparse matrix
sparse_matrix = lil_matrix((num_users, num_products), dtype=int)
product_index = {product: idx for idx, product in enumerate(unique_products)}

# Fill sparse matrix
for user_idx, seq in enumerate(sequences_for_mining):
    for product in seq:
        if product in product_index:
            sparse_matrix[user_idx, product_index[product]] = 1

# Convert to compressed format
sparse_matrix = sparse_matrix.tocsr()

print("Sparse matrix created successfully from sampled data!")


In [ ]:
min_support = 0.1  # Set a higher min_support for large datasets to reduce memory usage

# Apply sequential pattern mining
from pymining import seqmining

seq_patterns = seqmining.freq_seq_enum(sequences_for_mining, min_support)

# Extract and print frequent sequential patterns
print("Frequent Sequential Patterns:")
for pattern, support in seq_patterns:
    print(f"Pattern: {pattern}, Support: {support}")